#Gold Orchestration Logic

This notebook programmatically runs all Gold notebooks in sequence.
It becomes the single entry point for the Gold layer in Databricks Jobs. 

The code below turns the workspace table outputs into csv files. 

In [0]:
notebooks = [
    "./Gold_dim_customers",
    "./Gold_dim_products",
    "./Gold_fact_sales"
]

#Create a try and catch exception to see which notebook failed
for nb in notebooks:
    try:
        print(f"Running {nb}")
        dbutils.notebook.run(nb, timeout_seconds=0)
        print(f"✓ {nb} completed successfully")
    except Exception as e:
        print(f"✗ {nb} FAILED with error: {str(e)}")
        raise Exception(f"Notebook {nb} failed during execution. Original error: {str(e)}") from e

In [0]:
import os

# Define output directory for CSV files
output_dir = "/Workspace/Users/joesyby@gmail.com/databricks_lakehouse/Scripts/Gold/csv_exports"

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# List of tables to export
tables_to_export = [
    ("workspace.gold.dim_customers", "dim_customers.csv"),
    ("workspace.gold.dim_products", "dim_products.csv"),
    ("workspace.gold.fact_sales", "fact_sales.csv")
]

print(f"Exporting tables to: {output_dir}\n")

for table_name, csv_filename in tables_to_export:
    try:
        print(f"Exporting {table_name}...")
        
        # Read the table
        df = spark.table(table_name)
        
        # Convert to pandas and save as CSV
        pandas_df = df.toPandas()
        csv_path = f"{output_dir}/{csv_filename}"
        pandas_df.to_csv(csv_path, index=False)
        
        row_count = len(pandas_df)
        print(f"✓ {table_name} exported successfully ({row_count:,} rows) → {csv_filename}")
        
    except Exception as e:
        print(f"✗ Failed to export {table_name}: {str(e)}")

print(f"\nAll CSV files saved to: {output_dir}")